In [ ]:
!pip install chromadb google-generativeai pymupdf python-docx openpyxl langchain langchain-text-splitters

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DOCS_FOLDER = "/content/drive/MyDrive/Tech Internship Task Folder"

In [ ]:
import os
import time
import chromadb
import google.generativeai as genai
from pathlib import Path

# PDF
import fitz  # pymupdf

# DOCX
from docx import Document

# XLSX
import openpyxl

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure Gemini
from google.colab import userdata
from google import genai
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
def extract_pdf(path):
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

def extract_docx(path):
    doc = Document(path)
    return "\n".join(para.text for para in doc.paragraphs if para.text.strip())

def extract_xlsx(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    text = []
    for sheet in wb.worksheets:
        for row in sheet.iter_rows(values_only=True):
            row_text = " | ".join(str(cell) for cell in row if cell is not None)
            if row_text.strip():
                text.append(row_text)
    return "\n".join(text)

def extract_text(path):
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return extract_pdf(path)
    elif ext == ".docx":
        return extract_docx(path)
    elif ext == ".xlsx":
        return extract_xlsx(path)
    else:
        print(f"Skipping unsupported file: {path}")
        return None

In [ ]:
docs = []  # list of {"filename": ..., "text": ...}

for file in Path(DOCS_FOLDER).rglob("*"):
    if file.suffix.lower() in [".pdf", ".docx", ".xlsx"]:
        print(f"Processing: {file.name}")
        text = extract_text(str(file))
        if text and text.strip():
            docs.append({"filename": file.name, "text": text})

print(f"\nTotal documents loaded: {len(docs)}")

Processing: RIF Letterhead Template.docx
Processing: Self Declaration Letter.docx
Processing: Idobro profile for Impact Assessment.pdf
Processing: Previous Projects_with_Descriptions.xlsx
Processing: Follow up letter .docx
Processing: RIF Introduction V1 11092025.docx
Processing: Idobro introduction V1 11092025.docx
Processing: Idobro & RIF V1 07102025.docx
Processing: Idobro introduction.pdf
Processing: RIF Introduction.pdf
Processing: Declaration_Not generating IRN.docx
Processing: Company Profile.docx
Processing: Concept Note.docx
Processing: Key Projects of Idobro with Description (from Idobro website).docx
Processing: HG Proposal.docx
Processing: Sample Proposal Formatting Template.docx
Processing: Idobro Letterhead Template new.docx

Total documents loaded: 17


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=75,          # ~15% overlap, good for proposal docs
    separators=["\n\n", "\n", ".", " "]
)

chunks = []  # list of {"chunk_id": ..., "filename": ..., "text": ...}

for doc in docs:
    doc_chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['filename']}_chunk_{i}",
            "filename": doc["filename"],
            "text": chunk
        })

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 290


In [ ]:
def get_embedding(text, retries=3):
    for attempt in range(retries):
        try:
            result = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text
            )
            return result.embeddings[0].values
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(5)
    return None

In [ ]:
import time

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/idobro_db")
collection = chroma_client.get_or_create_collection(name="idobro_knowledge")

print("Embedding and storing chunks...")

for i, chunk in enumerate(chunks):

    success = False
    retries = 0

    while not success:
        try:
            embedding = get_embedding(chunk["text"])

            if embedding is None:
                raise Exception("Embedding returned None")

            collection.add(
                ids=[chunk["chunk_id"]],
                embeddings=[embedding],
                documents=[chunk["text"]],
                metadatas=[{"filename": chunk["filename"]}]
            )

            success = True

        except Exception as e:
            retries += 1

            # Extract retry time if available
            wait_time = 60  # default fallback

            if "retry in" in str(e).lower():
                try:
                    wait_time = int(str(e).split("retry in")[1].split("s")[0].strip())
                except:
                    pass

            print(f"Retry {retries} for chunk {i} after {wait_time}s...")
            time.sleep(wait_time + 2)  # buffer

    # Controlled pacing (important)
    if i % 10 == 0:
        print(f"  Stored {i+1}/{len(chunks)} chunks...")
        time.sleep(5)

print(f"\nDone! Total chunks in DB: {collection.count()}")

Embedding and storing chunks...
  Stored 1/290 chunks...
  Stored 11/290 chunks...
  Stored 21/290 chunks...
  Stored 31/290 chunks...
  Stored 41/290 chunks...
  Stored 51/290 chunks...
  Stored 61/290 chunks...
  Stored 71/290 chunks...
  Stored 81/290 chunks...
  Stored 91/290 chunks...
  Stored 101/290 chunks...
  Stored 111/290 chunks...
  Stored 121/290 chunks...
  Stored 131/290 chunks...
  Stored 141/290 chunks...
  Stored 151/290 chunks...
  Stored 161/290 chunks...
  Stored 171/290 chunks...
  Stored 181/290 chunks...
  Stored 191/290 chunks...
  Stored 201/290 chunks...
  Stored 211/290 chunks...
  Stored 221/290 chunks...
  Stored 231/290 chunks...
  Stored 241/290 chunks...
  Stored 251/290 chunks...
  Stored 261/290 chunks...
  Stored 271/290 chunks...
  Stored 281/290 chunks...

Done! Total chunks in DB: 290


In [ ]:
test_embedding = client.models.embed_content(
    model="gemini-embedding-001",
    contents="What sectors does Idobro work in?"
).embeddings[0].values

results = collection.query(
    query_embeddings=[test_embedding],
    n_results=3
)

print("Top 3 chunks retrieved for test query:\n")
for i, doc in enumerate(results["documents"][0]):
    print(f"--- Chunk {i+1} (from {results['metadatas'][0][i]['filename']}) ---")
    print(doc[:300])
    print()

Top 3 chunks retrieved for test query:

--- Chunk 1 (from Idobro profile for Impact Assessment.pdf) ---
1. Citizenship : Value-based programs for youth and Corporate employees 
2. Entrepreneurship : Support WSG Entrepreneurs (WSGE) as beneficiaries and enablers 
3. Partnership : Multi-stakeholder projects to leverage resources and scale development 
solutions for vulnerable groups and our environment.

--- Chunk 2 (from Company Profile.docx) ---
Idobro Impact Solutions
Overview
Idobro Impact Solutions is a social enterprise established in 2009, with a proven track record of executing projects in 7 countries and 24 states of India, impacting over 2.3 million individuals. 
We've partnered with esteemed organizations like Glenmark Foundation, 

--- Chunk 3 (from Company Profile.docx) ---
2. The GIFT Lens We apply the GIFT lens (Gender, Innovation, Finance, and Technology) to ensure that interventions are holistic and address the specific needs of diverse beneficiary groups, particularly w

In [ ]:
import shutil

shutil.copytree(
    "/content/idobro_db",
    "/content/drive/MyDrive/idobro_db",
    dirs_exist_ok=True
)

print("ChromaDB saved to Google Drive successfully!")

ChromaDB saved to Google Drive successfully!
